# Credibility Scoring Pipeline -- Walkthrough

iLab 14-01 / Decidr, Northfield Software Ltd pack.

This notebook runs the eight-stage pipeline described in the Group Proposal Report
end to end on the real Phase 1 + Phase 2 data, one claim at a time, and shows the
intermediate output of every stage rather than just a final number -- that
inspectability is the whole point of the project.

Stages: **ingest -> extract -> retrieve -> reason -> weight -> score -> calibrate -> explain**

Run top to bottom. Update `PHASE1_PATH` / `PHASE2_PATH` below if your data lives
somewhere else.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from src import ingest
from src.schema import Chunk, read_jsonl
from src.retrieve import BM25Retriever
from src.reason import classify_stance, reason_over_hits
from src.weight import apply_weights, independence_weights
from src.score import score_claim, bayesian_log_odds_score
from src.explain import explain_template
from src.pipeline import run_pipeline
from src import calibrate, evaluate
import json


## Stage 1 -- Ingestion

Parses the Phase 1 folder and the Phase 2 zip into one document/chunk store.
Phase 2 re-ships the Phase 1 policy/process/interview files unchanged, so
overlapping doc_ids are de-duplicated in favour of Phase 2 (see `src/ingest.py`
docstring) rather than double-counted as independent evidence.

In [2]:
PHASE1_PATH = "/sessions/wizardly-pensive-bell/mnt/Capstone/phase1_framework_development"
PHASE2_PATH = "/sessions/wizardly-pensive-bell/mnt/Capstone/phase2_model_development.zip"
OUT_DIR = "../outputs"

ingest.run(PHASE1_PATH, PHASE2_PATH, OUT_DIR)


phase1: 22 documents, 158 chunks (21 dropped as exact carry-overs already in phase2)
phase2: 44 documents, 739 chunks
TOTAL (deduplicated): 45 documents, 743 chunks -> ../outputs


In [3]:
chunks_rows = read_jsonl(os.path.join(OUT_DIR, "chunks.jsonl"))
chunks = [Chunk(**{k: v for k, v in r.items() if k in Chunk.__dataclass_fields__}) for r in chunks_rows]

import pandas as pd
docs_df = pd.read_json(os.path.join(OUT_DIR, "documents.jsonl"), lines=True)
docs_df.groupby("source_type").size().sort_values(ascending=False)


source_type
meeting            10
interview          10
policy              8
process_doc         7
overview            5
audit_log           1
comms               1
crm                 1
labelled_sample     1
tasks               1
dtype: int64

## Stage 2 -- Claim extraction (rule-based baseline)

A quick look at what the modal-verb heuristic proposes. These are **unreviewed
candidates** -- per the proposal's scope, none of these should be trusted
without a human pass.

In [4]:
from src.extract import extract_claims_rule_based
candidate_claims = extract_claims_rule_based(chunks)
print(f"{len(candidate_claims)} candidate claims proposed from {len(chunks)} chunks\n")
for c in candidate_claims[:5]:
    print(f"- [{c.layer}] {c.claim_text}  (from {c.origin_doc_id})")


18 candidate claims proposed from 743 chunks

- [unclassified] claim id: C01; claim category: Financial control; reference claim: Discounts above 15% require Finance (Head of Finance & People Ops) approval before the deal is closed.; reference status: Formal truth; valid from: 2025 03 01; responsible role: Head of Finance & People Ops; known exceptions: No formally documented exceptions.; expected supporting sources: POL 01; INT 02; INT 03; MTG 01; difficulty level: Low  (from LABELLED-CLAIMS-SAMPLE)
- [unclassified] Production deployments touching customer billing require his sign off before release.  (from ROLE-DESCRIPTIONS)
- [formal] Any discount of 15% or greater off list price on a new business deal requires written approval from the Head of Finance & People Ops before the deal is marked Closed Won in the CRM.  (from POL-01)
- [formal] Approval must be recorded in the CRM's approval record field, referencing the approver and date.  (from POL-01)
- [formal] Any deviation from the 

## Stages 3-8 for one claim: the discount-approval example

This is the claim the whole team has used since Week 2/3: *"Discounts above
15% require Finance approval before the deal is closed."* We already know
independently (from the data-exploration write-up) which documents should
turn up: POL-01 (current, supports it), PROC-02 (stale title, minor
contradiction), and the Diane Okafor / Slack workaround trail (INT-02, INT-04,
COMMS-01..04) which complicates it in practice.

In [5]:
retriever = BM25Retriever(chunks)

claim_text = "Discounts above 15 percent require Finance Head of Finance and People Ops approval before the deal is closed."
claim_id = "C01"

hits = retriever.search(claim_text, top_k=15)
print(f"Stage 3 -- Retrieval: {len(hits)} chunks retrieved\n")
for h in hits[:8]:
    print(f"  [{h.score:5.2f}] {h.doc_id:22s} ({h.source_type})")


Stage 3 -- Retrieval: 15 chunks retrieved

  [38.08] LABELLED-CLAIMS-SAMPLE (labelled_sample)
  [27.47] POL-02                 (policy)
  [27.41] POL-01                 (policy)
  [25.44] POL-07                 (policy)
  [23.42] ROLE-DESCRIPTIONS      (overview)
  [22.68] PROC-02                (process_doc)
  [20.56] INT-03                 (interview)
  [20.42] PROC-04                (process_doc)


In [6]:
reasoned = reason_over_hits(claim_text, hits)
print(f"Stage 4 -- Reasoning: {len(reasoned)} chunks kept after dropping 'unrelated'\n")
for r in reasoned:
    print(f"  {r['stance']:11s} conf={r['stance_confidence']:.2f}  {r['doc_id']}")


Stage 4 -- Reasoning: 15 chunks kept after dropping 'unrelated'

  contradict  conf=0.75  LABELLED-CLAIMS-SAMPLE
  contradict  conf=0.75  POL-02
  contradict  conf=0.75  POL-01
  support     conf=0.80  POL-07
  contradict  conf=0.75  ROLE-DESCRIPTIONS
  support     conf=0.80  PROC-02
  support     conf=0.80  INT-03
  support     conf=0.80  PROC-04
  support     conf=0.80  MTG-01
  support     conf=0.80  PROC-04
  contradict  conf=0.75  ROLE-DESCRIPTIONS
  contradict  conf=0.73  COMPANY-OVERVIEW
  support     conf=0.73  MTG-09
  contradict  conf=0.73  ROLE-DESCRIPTIONS
  contradict  conf=0.75  COMMUNICATIONS-LOG


**A known limitation, worth reading rather than skipping past.** POL-01 --
the current, approved policy that states this exact rule -- gets flagged
`contradict` above. Why: the heuristic in `src/reason.py` scans the whole
retrieved chunk for negation/contrast words like *"without"*, and POL-01's
chunk happens to contain *"Discounts below 15% may be approved ... without
further sign-off"* -- a different sub-rule, about sub-15% deals, sitting in
the same paragraph as the 15%+ rule this claim is actually about.

This is exactly the gap Section 2.1 of the proposal names: lexical-overlap
heuristics are not a stance classifier, and the plan is to compare this
baseline against a small NLI model and a constrained LLM prompt before
trusting either one. Leave this cell as a live demonstration of *why* --
don't quietly patch the heuristic to hide it.

In [7]:
superseded_lookup = {c.doc_id: c.metadata.get("superseded", False) for c in chunks}
weighted = apply_weights(reasoned, superseded_lookup=superseded_lookup)

print("Stage 5 -- Weighting\n")
for w in sorted(weighted, key=lambda x: -x["final_weight"]):
    print(f"  {w['doc_id']:22s} stance={w['stance']:11s} tier={w['authority_tier']:18s} "
          f"indep={w['independence_weight']:.2f}  final_weight={w['final_weight']:.3f}")


Stage 5 -- Weighting

  POL-07                 stance=support     tier=approved-formal    indep=1.00  final_weight=0.800
  POL-01                 stance=contradict  tier=approved-formal    indep=1.00  final_weight=0.750
  PROC-02                stance=support     tier=approved-formal    indep=1.00  final_weight=0.640
  PROC-04                stance=support     tier=approved-formal    indep=1.00  final_weight=0.640
  PROC-04                stance=support     tier=approved-formal    indep=1.00  final_weight=0.640
  ROLE-DESCRIPTIONS      stance=contradict  tier=secondhand         indep=1.00  final_weight=0.562
  ROLE-DESCRIPTIONS      stance=contradict  tier=secondhand         indep=1.00  final_weight=0.562
  COMPANY-OVERVIEW       stance=contradict  tier=secondhand         indep=1.00  final_weight=0.547
  ROLE-DESCRIPTIONS      stance=contradict  tier=secondhand         indep=1.00  final_weight=0.547
  MTG-01                 stance=support     tier=informal-testimony indep=1.00  final_w

In [8]:
scored = score_claim(weighted, prior=0.5, method="bayesian")
print("Stage 6 -- Scoring\n")
print(json.dumps(scored, indent=2))


Stage 6 -- Scoring

{
  "credibility_score": 0.5611,
  "credible_interval": [
    0.0347,
    0.9672
  ],
  "status": "Conflicted",
  "confidence": "High",
  "supporting_evidence": [
    "POL-07::2",
    "PROC-02::1",
    "INT-03::0",
    "PROC-04::4",
    "MTG-01::0",
    "PROC-04::3",
    "MTG-09::0"
  ],
  "contradicting_evidence": [
    "LABELLED-CLAIMS-SAMPLE::0",
    "POL-02::2",
    "POL-01::2",
    "ROLE-DESCRIPTIONS::11",
    "ROLE-DESCRIPTIONS::3",
    "COMPANY-OVERVIEW::4",
    "ROLE-DESCRIPTIONS::12",
    "COMMUNICATIONS-LOG::5"
  ]
}


In [9]:
explanation = explain_template(claim_text, scored, weighted)
print("Stage 8 -- Explanation\n")
print(explanation)


Stage 8 -- Explanation

Claim: "Discounts above 15 percent require Finance Head of Finance and People Ops approval before the deal is closed." Status: Conflicted. Credibility score: 0.56 (range 0.03-0.97). Confidence: High. Raised by: POL-07 (weight 0.80), PROC-02 (weight 0.64), PROC-04 (weight 0.64). Lowered by: POL-01 (weight 0.75), ROLE-DESCRIPTIONS (weight 0.56), ROLE-DESCRIPTIONS (weight 0.56). This claim is marked Conflicted because meaningful weight sits on both sides. That is the honest answer on this pack, not a failure of the method. An extra approved, dated, first-hand document naming this exact rule would move this status the most.


## Stage 7 -- Calibration (guard demonstration)

We do not have anywhere near enough labelled claims yet. This cell shows the
guard in `src/calibrate.py` correctly refusing to compute a Brier score / ECE
on too small a set, rather than producing a number that looks precise but
means nothing.

In [10]:
ok, msg = calibrate.enough_labels_to_calibrate(n_labelled=4)
print(msg)


Only 4 labelled claims available (need >= 30). Calibration has not been run; treat any stated confidence level as unverified.


## Whole pipeline, one call: `run_pipeline`

Everything above, wrapped into a single function call per claim -- this is
what `src/pipeline.py`'s CLI and any future Streamlit demo page should call.

In [11]:
result = run_pipeline(claim_text, claim_id, retriever)
sc = result["scored_claim"]
print(f"{sc.claim_id}: {sc.status}  (score={sc.credibility_score}, confidence={sc.confidence})")


C01: Conflicted  (score=0.5611, confidence=High)


## Gold test claims: retrieval recall + status agreement

`data/gold_claims.json` holds a handful of claims the team already
understands deeply from the data-exploration phase, including the
**Ridgeway renewal (C03)** -- the one genuinely unresolved case flagged
since Week 2, included specifically to test whether the pipeline can land
on *Not enough evidence / Conflicted* instead of confidently inventing an
answer. This is a sanity check on retrieval and status behaviour, not a
calibration run (four to six examples is nowhere near the 30+ needed for
that -- see the guard above).

In [12]:
evaluate.run(os.path.join(OUT_DIR, "chunks.jsonl"), "../data/gold_claims.json")


claim_id   retrieval_recall  status (expected -> actual)             
----------------------------------------------------------------------


C01        0.333                     Conflicted -> Conflicted         [OK]


C02        1.0                       Conflicted -> Unsupported        [CHECK]


C03        1.0               Not enough evidence -> Unsupported        [CHECK]


C04        1.0                       Conflicted -> Unsupported        [CHECK]


## Reading the table above

- **Recall below 1.0 on C01** means the keyword baseline is not finding every
  document the team already knows is relevant (the Slack/INT evidence in
  particular) -- exactly the retrieval question Section 3.2 asks, and a real
  measurement rather than an assumption.
- **Status mismatches marked `[CHECK]`** are not a broken notebook -- they are
  the current, honest state of a rule-based reasoning/weighting baseline that
  the proposal always intended to replace piece by piece. Track these here as
  the team upgrades stage 4 (reasoning) and stage 2 (extraction confidence)
  over the coming weeks.

## Next steps per role (see the Group Proposal Report, Section 5.2)

- **Bhavika** -- extend `src/ingest.py`: parse real dates out of each
  document's metadata block so `recency_weight` in `src/weight.py` stops being
  a placeholder 1.0.
- **Nakul** -- replace/extend `extract_claims_rule_based` with an
  LLM-assisted extractor behind `extract_claims_llm`, keeping the
  role+action+condition contract.
- **Rohan** -- measure BM25 recall properly (this notebook's evaluate table is
  a start) and decide whether `SemanticRetriever` earns its dependency.
- **Shreyash** -- replace the lexical-overlap heuristic in `src/reason.py`
  with a real NLI model or constrained LLM prompt, compared against hand
  labels.
- **Prathamesh** -- grow `data/gold_claims.json` well past 30 labelled claims,
  then run `src/calibrate.py` for real.